# Proyecto Módulo 4 - Preparación de Datos
## Lección 1 - Generación de datos con NumPy

In [3]:
import numpy as np

# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================
SEED = 13094978
np.random.seed(SEED)

N_CLIENTES = 300
N_TRANSACCIONES = 1200

RUTA_CLIENTES = "../data/raw/clientes_base.npy"
RUTA_TRANSACCIONES = "../data/raw/transacciones_base.npy"


# =========================================================
# FUNCIONES AUXILIARES
# =========================================================
def generar_fechas_aleatorias(fecha_inicio, fecha_fin, n):
    """
    Genera fechas aleatorias entre dos fechas usando numpy.datetime64.
    """
    inicio = np.datetime64(fecha_inicio, "D")
    fin = np.datetime64(fecha_fin, "D")
    dias_disponibles = (fin - inicio).astype(int)
    desplazamientos = np.random.randint(0, dias_disponibles + 1, size=n)
    return inicio + desplazamientos.astype("timedelta64[D]")


# =========================================================
# GENERACIÓN DE CLIENTES
# =========================================================
def generar_clientes(n_clientes=300):
    nombres_pool = np.array([
        "Juan", "María", "Pedro", "Camila", "Diego", "Valentina",
        "Sofía", "Matías", "Fernanda", "Javiera", "Benjamín",
        "Antonia", "Vicente", "Catalina", "Tomás", "Daniela"
    ])

    apellidos_pool = np.array([
        "González", "Muñoz", "Rojas", "Díaz", "Pérez", "Soto",
        "Contreras", "Silva", "Martínez", "Sepúlveda", "Morales",
        "Rodríguez", "López", "Fuentes", "Hernández", "Torres"
    ])

    ciudades_pool = np.array([
        "Santiago", "Valparaíso", "Concepción", "La Serena",
        "Antofagasta", "Temuco", "Rancagua", "Puerto Montt"
    ], dtype=object)

    categorias_pool = np.array([
        "Tecnología", "Hogar", "Moda", "Deportes", "Belleza", "Libros"
    ], dtype=object)

    ids_clientes = np.arange(1001, 1001 + n_clientes)

    nombres = np.random.choice(nombres_pool, size=n_clientes)
    apellidos = np.random.choice(apellidos_pool, size=n_clientes)
    nombre_completo = np.char.add(np.char.add(nombres, " "), apellidos)

    edad = np.random.randint(18, 70, size=n_clientes).astype(float)
    ciudad = np.random.choice(ciudades_pool, size=n_clientes).astype(object)
    fecha_registro = generar_fechas_aleatorias("2022-01-01", "2024-12-31", n_clientes)
    categoria_favorita = np.random.choice(categorias_pool, size=n_clientes).astype(object)

    ingreso_estimado = np.random.normal(loc=850000, scale=250000, size=n_clientes)
    ingreso_estimado = np.clip(ingreso_estimado, 300000, None)
    ingreso_estimado = np.round(ingreso_estimado, 0)

    idx_nulos_edad = np.random.choice(n_clientes, size=12, replace=False)
    edad[idx_nulos_edad] = np.nan

    idx_nulos_ciudad = np.random.choice(n_clientes, size=8, replace=False)
    ciudad[idx_nulos_ciudad] = None

    idx_ciudad_inconsistente = np.random.choice(n_clientes, size=9, replace=False)
    ciudad[idx_ciudad_inconsistente[:3]] = "santiago"
    ciudad[idx_ciudad_inconsistente[3:6]] = "Valparaíso "
    ciudad[idx_ciudad_inconsistente[6:]] = "CONCEPCIÓN"

    idx_categoria_inconsistente = np.random.choice(n_clientes, size=6, replace=False)
    categoria_favorita[idx_categoria_inconsistente[:2]] = "tecnologia"
    categoria_favorita[idx_categoria_inconsistente[2:4]] = "HOGAR"
    categoria_favorita[idx_categoria_inconsistente[4:]] = "moda "

    clientes = {
        "id_cliente": ids_clientes,
        "nombre": nombre_completo,
        "edad": edad,
        "ciudad": ciudad,
        "fecha_registro": fecha_registro,
        "categoria_favorita": categoria_favorita,
        "ingreso_estimado": ingreso_estimado
    }

    return clientes


# =========================================================
# GENERACIÓN DE TRANSACCIONES
# =========================================================
def generar_transacciones(clientes, n_transacciones=1200):
    categorias_producto_pool = np.array([
        "Tecnología", "Hogar", "Moda", "Deportes", "Belleza", "Libros"
    ], dtype=object)

    canales_pool = np.array([
        "Web", "App", "Marketplace"
    ], dtype=object)

    ids_clientes = clientes["id_cliente"]
    n_clientes = len(ids_clientes)

    indices_clientes = np.random.randint(0, n_clientes, size=n_transacciones)
    id_cliente_tx = ids_clientes[indices_clientes]

    id_transaccion = np.arange(50001, 50001 + n_transacciones)
    fecha_transaccion = generar_fechas_aleatorias("2024-01-01", "2024-12-31", n_transacciones)
    categoria_producto = np.random.choice(categorias_producto_pool, size=n_transacciones).astype(object)
    cantidad = np.random.randint(1, 6, size=n_transacciones)

    precio_unitario = np.random.uniform(5000, 250000, size=n_transacciones)
    precio_unitario = np.round(precio_unitario, 2)

    descuento_pct = np.random.choice(
        [0.00, 0.05, 0.10, 0.15, 0.20],
        size=n_transacciones,
        p=[0.30, 0.25, 0.20, 0.15, 0.10]
    ).astype(float)

    canal_venta = np.random.choice(canales_pool, size=n_transacciones).astype(object)

    monto_compra = cantidad * precio_unitario * (1 - descuento_pct)
    monto_compra = np.round(monto_compra, 2)

    idx_nulos_descuento = np.random.choice(n_transacciones, size=20, replace=False)
    descuento_pct[idx_nulos_descuento] = np.nan

    idx_outliers = np.random.choice(n_transacciones, size=10, replace=False)
    precio_unitario[idx_outliers] = np.round(precio_unitario[idx_outliers] * 5, 2)
    monto_compra[idx_outliers] = np.round(monto_compra[idx_outliers] * 8, 2)

    idx_categorias_inconsistentes = np.random.choice(n_transacciones, size=9, replace=False)
    categoria_producto[idx_categorias_inconsistentes[:3]] = "tecnologia"
    categoria_producto[idx_categorias_inconsistentes[3:6]] = "HOGAR"
    categoria_producto[idx_categorias_inconsistentes[6:]] = "moda "

    idx_canales_inconsistentes = np.random.choice(n_transacciones, size=6, replace=False)
    canal_venta[idx_canales_inconsistentes[:2]] = "web"
    canal_venta[idx_canales_inconsistentes[2:4]] = "APP"
    canal_venta[idx_canales_inconsistentes[4:]] = "Marketplace "

    transacciones = {
        "id_transaccion": id_transaccion,
        "id_cliente": id_cliente_tx,
        "fecha_transaccion": fecha_transaccion,
        "categoria_producto": categoria_producto,
        "cantidad": cantidad,
        "precio_unitario": precio_unitario,
        "descuento_pct": descuento_pct,
        "canal_venta": canal_venta,
        "monto_compra": monto_compra
    }

    idx_duplicados = np.random.choice(len(transacciones["id_transaccion"]), size=12, replace=False)

    for clave in transacciones:
        transacciones[clave] = np.concatenate([
            transacciones[clave],
            transacciones[clave][idx_duplicados]
        ])

    return transacciones


# =========================================================
# RESUMEN BÁSICO CON NUMPY
# =========================================================
def resumen_numpy(clientes, transacciones):
    total_clientes = len(clientes["id_cliente"])
    total_transacciones = len(transacciones["id_transaccion"])

    edad_promedio = np.nanmean(clientes["edad"])
    ingreso_promedio = np.nanmean(clientes["ingreso_estimado"])
    total_ventas = np.nansum(transacciones["monto_compra"])
    ticket_promedio = np.nanmean(transacciones["monto_compra"])

    categorias, conteos_categorias = np.unique(
        transacciones["categoria_producto"].astype(str),
        return_counts=True
    )

    print("===== RESUMEN LECCIÓN 1 =====")
    print(f"Total de clientes generados: {total_clientes}")
    print(f"Total de transacciones generadas: {total_transacciones}")
    print(f"Edad promedio de clientes: {edad_promedio:.2f}")
    print(f"Ingreso estimado promedio: {ingreso_promedio:.0f}")
    print(f"Ventas totales: {total_ventas:.2f}")
    print(f"Ticket promedio: {ticket_promedio:.2f}")
    print("\nConteo por categoría de producto:")
    for cat, cnt in zip(categorias, conteos_categorias):
        print(f" - {cat}: {cnt}")


# =========================================================
# GUARDADO DE ARCHIVOS
# =========================================================
def guardar_bases(clientes, transacciones,
                  ruta_clientes="../data/raw/clientes_base.npy",
                  ruta_transacciones="../data/raw/transacciones_base.npy"):
    np.save(ruta_clientes, clientes, allow_pickle=True)
    np.save(ruta_transacciones, transacciones, allow_pickle=True)
    print(f"Archivo guardado: {ruta_clientes}")
    print(f"Archivo guardado: {ruta_transacciones}")


# =========================================================
# EJECUCIÓN PRINCIPAL
# =========================================================
clientes_base = generar_clientes(N_CLIENTES)
transacciones_base = generar_transacciones(clientes_base, N_TRANSACCIONES)

resumen_numpy(clientes_base, transacciones_base)
guardar_bases(clientes_base, transacciones_base, RUTA_CLIENTES, RUTA_TRANSACCIONES)

print("\nPrimeros 5 id_cliente:", clientes_base["id_cliente"][:5])
print("Primeras 5 edades:", clientes_base["edad"][:5])
print("Primeras 5 transacciones:", transacciones_base["id_transaccion"][:5])
print("Primeros 5 montos:", transacciones_base["monto_compra"][:5])

===== RESUMEN LECCIÓN 1 =====
Total de clientes generados: 300
Total de transacciones generadas: 1212
Edad promedio de clientes: 43.43
Ingreso estimado promedio: 846598
Ventas totales: 461414793.25
Ticket promedio: 380705.27

Conteo por categoría de producto:
 - Belleza: 189
 - Deportes: 201
 - HOGAR: 3
 - Hogar: 177
 - Libros: 199
 - Moda: 230
 - Tecnología: 207
 - moda : 3
 - tecnologia: 3
Archivo guardado: ../data/raw/clientes_base.npy
Archivo guardado: ../data/raw/transacciones_base.npy

Primeros 5 id_cliente: [1001 1002 1003 1004 1005]
Primeras 5 edades: [41. 33. 59. 44. 24.]
Primeras 5 transacciones: [50001 50002 50003 50004 50005]
Primeros 5 montos: [239633.56  39957.39 458615.39  80071.35 299587.54]


## Lección 2 - Exploración y transformación inicial con Pandas

In [4]:
import pandas as pd
import numpy as np

# =========================================================
# CARGA DE ARCHIVOS GENERADOS EN LA LECCIÓN 1
# =========================================================
RUTA_CLIENTES_NPY = "../data/raw/clientes_base.npy"
RUTA_TRANSACCIONES_NPY = "../data/raw/transacciones_base.npy"

clientes_cargados = np.load(RUTA_CLIENTES_NPY, allow_pickle=True).item()
transacciones_cargadas = np.load(RUTA_TRANSACCIONES_NPY, allow_pickle=True).item()

# =========================================================
# CONVERSIÓN A DATAFRAME
# =========================================================
df_clientes = pd.DataFrame(clientes_cargados)
df_transacciones = pd.DataFrame(transacciones_cargadas)

# Convertir fechas a formato fecha reconocible por pandas
df_clientes["fecha_registro"] = pd.to_datetime(df_clientes["fecha_registro"])
df_transacciones["fecha_transaccion"] = pd.to_datetime(df_transacciones["fecha_transaccion"])

# =========================================================
# CREACIÓN DE DATAFRAME PRELIMINAR UNIFICADO
# =========================================================
df_preliminar = df_transacciones.merge(
    df_clientes,
    on="id_cliente",
    how="left"
)

# =========================================================
# EXPLORACIÓN INICIAL
# =========================================================
print("===== PRIMERAS 5 FILAS DE CLIENTES =====")
display(df_clientes.head())

print("===== ÚLTIMAS 5 FILAS DE CLIENTES =====")
display(df_clientes.tail())

print("===== PRIMERAS 5 FILAS DE TRANSACCIONES =====")
display(df_transacciones.head())

print("===== ÚLTIMAS 5 FILAS DE TRANSACCIONES =====")
display(df_transacciones.tail())

print("===== PRIMERAS 5 FILAS DEL DATAFRAME PRELIMINAR =====")
display(df_preliminar.head())

print("===== INFORMACIÓN GENERAL DE CLIENTES =====")
df_clientes.info()

print("\n===== INFORMACIÓN GENERAL DE TRANSACCIONES =====")
df_transacciones.info()

print("\n===== INFORMACIÓN GENERAL DEL DATAFRAME PRELIMINAR =====")
df_preliminar.info()

print("\n===== ESTADÍSTICAS DESCRIPTIVAS =====")
display(df_preliminar.describe(include="all"))

# =========================================================
# FILTROS CONDICIONALES
# =========================================================
print("\n===== TRANSACCIONES CON MONTO MAYOR A 500000 =====")
display(df_preliminar[df_preliminar["monto_compra"] > 500000].head(10))

print("\n===== CLIENTES DE SANTIAGO =====")
display(df_preliminar[df_preliminar["ciudad"] == "Santiago"].head(10))

print("\n===== CLIENTES CON EDAD MAYOR A 50 =====")
display(df_preliminar[df_preliminar["edad"] > 50].head(10))

print("\n===== TRANSACCIONES DEL CANAL WEB =====")
display(df_preliminar[df_preliminar["canal_venta"] == "Web"].head(10))

# =========================================================
# EXPORTACIÓN DEL CSV PRELIMINAR
# =========================================================
RUTA_PRELIMINAR_CSV = "../data/interim/dataset_preliminar.csv"
df_preliminar.to_csv(RUTA_PRELIMINAR_CSV, index=False)

print(f"\nArchivo CSV preliminar guardado en: {RUTA_PRELIMINAR_CSV}")
print(f"Filas del dataset preliminar: {df_preliminar.shape[0]}")
print(f"Columnas del dataset preliminar: {df_preliminar.shape[1]}")

===== PRIMERAS 5 FILAS DE CLIENTES =====


,id_cliente,nombre,edad,ciudad,fecha_registro,categoria_favorita,ingreso_estimado
0,1001,Diego Pérez,41.0,Puerto Montt,2023-08-28,Libros,650235.0
1,1002,Juan Contreras,33.0,Santiago,2024-04-09,Belleza,850503.0
2,1003,Sofía Sepúlveda,59.0,Antofagasta,2024-01-22,Belleza,1255397.0
3,1004,Antonia López,44.0,Antofagasta,2023-01-26,Deportes,765369.0
4,1005,Camila Hernández,24.0,La Serena,2023-07-05,Belleza,696195.0


===== ÚLTIMAS 5 FILAS DE CLIENTES =====


,id_cliente,nombre,edad,ciudad,fecha_registro,categoria_favorita,ingreso_estimado
295,1296,Diego Martínez,49.0,Antofagasta,2024-02-22,Libros,933572.0
296,1297,Diego Rojas,49.0,La Serena,2023-03-20,Belleza,1033414.0
297,1298,Camila Díaz,39.0,Valparaíso,2023-11-27,Tecnología,924876.0
298,1299,Valentina Morales,31.0,Valparaíso,2023-01-31,Hogar,300000.0
299,1300,Antonia Silva,40.0,Santiago,2023-01-19,Moda,1039059.0


===== PRIMERAS 5 FILAS DE TRANSACCIONES =====


...


===== CLIENTES CON EDAD MAYOR A 50 =====


,id_transaccion,id_cliente,fecha_transaccion,categoria_producto,cantidad,precio_unitario,descuento_pct,canal_venta,monto_compra,nombre,edad,ciudad,fecha_registro,categoria_favorita,ingreso_estimado
4,50005,1112,2024-06-21,Moda,5,66575.01,0.10,App,299587.54,Vicente Rojas,52.0,Santiago,2024-10-20,Libros,514152.0
6,50007,1030,2024-05-18,Hogar,3,29677.98,0.15,Web,75678.85,Daniela Rojas,62.0,Valparaíso,2024-05-17,Deportes,910769.0
11,50012,1141,2024-04-23,Libros,3,53302.00,0.10,App,143915.40,Matías Fuentes,56.0,Temuco,2023-03-04,Hogar,777672.0
14,50015,1090,2024-09-23,Libros,5,226702.84,0.00,Marketplace,1133514.20,Diego González,60.0,Concepción,2023-08-04,Libros,805526.0
23,50024,1240,2024-07-16,Deportes,5,24165.53,0.15,App,102703.50,Antonia Fuentes,60.0,Valparaíso,2022-09-24,Belleza,1130335.0
25,50026,1283,2024-07-14,Libros,3,185196.63,0.15,Web,472251.41,Fernanda Martínez,59.0,Valparaíso,2023-02-22,Libros,820956.0
33,50034,1234,2024-05-19,Libros,4,25857.41,0.20,App,82743.71,Juan Hernández,56.0,Temuco,2023-10-26,Moda,880253.0
34,50035,1068,2024-11-09,Libros,2,84525.93,0.05,Marketplace,160599.27,Benjamín Hernández,56.0,Concepción,2024-02-04,Belleza,1158712.0
35,50036,1257,2024-11-20,Libros,1,68223.08,0.00,Web,68223.08,Daniela Díaz,54.0,Santiago,2023-10-10,Belleza,532963.0
36,50037,1068,2024-10-10,Moda,3,76653.27,0.10,Web,206963.83,Benjamín Hernández,56.0,Concepción,2024-02-04,Belleza,1158712.0



===== TRANSACCIONES DEL CANAL WEB =====


,id_transaccion,id_cliente,fecha_transaccion,categoria_producto,cantidad,precio_unitario,descuento_pct,canal_venta,monto_compra,nombre,edad,ciudad,fecha_registro,categoria_favorita,ingreso_estimado
0,50001,1294,2024-01-20,Moda,4,70480.46,0.15,Web,239633.56,Diego Martínez,NaN,Santiago,2024-11-13,Libros,855625.0
2,50003,1222,2024-05-31,Hogar,2,241376.52,0.05,Web,458615.39,Vicente Rojas,44.0,Temuco,2022-12-06,Hogar,1224321.0
5,50006,1076,2024-06-07,Deportes,2,53614.88,0.10,Web,96506.78,Camila López,44.0,Puerto Montt,2024-04-02,Deportes,721760.0
6,50007,1030,2024-05-18,Hogar,3,29677.98,0.15,Web,75678.85,Daniela Rojas,62.0,Valparaíso,2024-05-17,Deportes,910769.0
7,50008,1063,2024-01-04,Tecnología,4,200798.11,0.05,Web,763032.82,Diego Rodríguez,19.0,Santiago,2024-09-28,Moda,851974.0
13,50014,1183,2024-12-30,Deportes,1,192489.77,0.05,Web,182865.28,Juan Hernández,24.0,Valparaíso,2022-12-15,Tecnología,871378.0
21,50022,1215,2024-01-10,Hogar,1,65720.32,0.10,Web,59148.29,Sofía Díaz,28.0,Rancagua,2022-09-22,Libros,574547.0
22,50023,1187,2024-05-15,Moda,3,144942.75,0.00,Web,434828.25,Sofía Contreras,50.0,Santiago,2022-08-08,Tecnología,1085456.0
24,50025,1027,2024-05-20,Moda,5,102373.76,0.00,Web,511868.80,Pedro Morales,25.0,CONCEPCIÓN,2024-08-21,Moda,618112.0
25,50026,1283,2024-07-14,Libros,3,185196.63,0.15,Web,472251.41,Fernanda Martínez,59.0,Valparaíso,2023-02-22,Libros,820956.0



Archivo CSV preliminar guardado en: ../data/interim/dataset_preliminar.csv
Filas del dataset preliminar: 1212
Columnas del dataset preliminar: 15


## Lección 3 - Obtención de datos desde archivos

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata

# =========================================================
# CONFIGURACIÓN DE RUTAS
# =========================================================
RUTA_CSV_PRELIMINAR = "../data/interim/dataset_preliminar.csv"
RUTA_EXCEL_COMPLEMENTARIO = "../data/raw/clientes_complementario.xlsx"
RUTA_HTML_WEB = "../data/raw/tabla_ciudades.html"
RUTA_CONSOLIDADO = "../data/interim/dataset_consolidado.csv"

# =========================================================
# FUNCIÓN AUXILIAR PARA NORMALIZAR TEXTO
# =========================================================
def normalizar_texto(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    return texto

# =========================================================
# 1) CARGAR EL CSV GENERADO EN LA LECCIÓN 2
# =========================================================
df_preliminar = pd.read_csv(RUTA_CSV_PRELIMINAR)

# Reconversión de fechas
df_preliminar["fecha_transaccion"] = pd.to_datetime(df_preliminar["fecha_transaccion"])
df_preliminar["fecha_registro"] = pd.to_datetime(df_preliminar["fecha_registro"])

print("===== DATAFRAME PRELIMINAR CARGADO DESDE CSV =====")
display(df_preliminar.head())
print(f"Dimensiones del dataset preliminar: {df_preliminar.shape}")

# =========================================================
# 2) CREAR UN EXCEL COMPLEMENTARIO Y LEERLO
# =========================================================
np.random.seed(13094978)

df_excel_base = df_preliminar[["id_cliente"]].drop_duplicates().copy()

segmentos = ["Bronce", "Plata", "Oro", "Premium"]
canales_adquisicion = ["Google Ads", "Instagram", "Facebook", "Referido", "Orgánico"]
metodos_pago = ["Tarjeta crédito", "Tarjeta débito", "Transferencia", "PayPal"]

df_excel_base["segmento_cliente"] = np.random.choice(segmentos, size=len(df_excel_base))
df_excel_base["canal_adquisicion"] = np.random.choice(canales_adquisicion, size=len(df_excel_base))
df_excel_base["metodo_pago_preferido"] = np.random.choice(metodos_pago, size=len(df_excel_base))
df_excel_base["score_fidelizacion"] = np.random.randint(40, 101, size=len(df_excel_base))

# Introducción intencional de algunos nulos e inconsistencias
idx_nulos_score = np.random.choice(df_excel_base.index, size=10, replace=False)
df_excel_base.loc[idx_nulos_score, "score_fidelizacion"] = np.nan

idx_inconsistencias_segmento = np.random.choice(df_excel_base.index, size=6, replace=False)
df_excel_base.loc[idx_inconsistencias_segmento[:2], "segmento_cliente"] = "premium"
df_excel_base.loc[idx_inconsistencias_segmento[2:4], "segmento_cliente"] = "ORO"
df_excel_base.loc[idx_inconsistencias_segmento[4:], "segmento_cliente"] = "plata "

# Guardar Excel
df_excel_base.to_excel(RUTA_EXCEL_COMPLEMENTARIO, index=False)

# Leer Excel
df_excel = pd.read_excel(RUTA_EXCEL_COMPLEMENTARIO)

print("\n===== FUENTE EXCEL COMPLEMENTARIA =====")
display(df_excel.head())
print(f"Dimensiones del Excel complementario: {df_excel.shape}")

# =========================================================
# 3) CREAR UNA TABLA HTML Y LEERLA CON read_html()
# =========================================================
html_tabla = """
<table>
    <thead>
        <tr>
            <th>ciudad</th>
            <th>region</th>
            <th>macrozona</th>
        </tr>
    </thead>
    <tbody>
        <tr><td>Santiago</td><td>Región Metropolitana</td><td>Centro</td></tr>
        <tr><td>Valparaíso</td><td>Región de Valparaíso</td><td>Centro</td></tr>
        <tr><td>Concepción</td><td>Región del Biobío</td><td>Sur</td></tr>
        <tr><td>La Serena</td><td>Región de Coquimbo</td><td>Norte</td></tr>
        <tr><td>Antofagasta</td><td>Región de Antofagasta</td><td>Norte</td></tr>
        <tr><td>Temuco</td><td>Región de La Araucanía</td><td>Sur</td></tr>
        <tr><td>Rancagua</td><td>Región de O'Higgins</td><td>Centro</td></tr>
        <tr><td>Puerto Montt</td><td>Región de Los Lagos</td><td>Sur</td></tr>
    </tbody>
</table>
"""

with open(RUTA_HTML_WEB, "w", encoding="utf-8") as archivo_html:
    archivo_html.write(html_tabla)

df_web = pd.read_html(RUTA_HTML_WEB)[0]

print("\n===== TABLA WEB LEÍDA CON read_html() =====")
display(df_web)
print(f"Dimensiones de la tabla web: {df_web.shape}")

# =========================================================
# 4) UNIFICAR LAS FUENTES
# =========================================================
# Normalizar ciudad para poder unir pese a mayúsculas, espacios y tildes
df_preliminar["ciudad_normalizada"] = df_preliminar["ciudad"].apply(normalizar_texto)
df_web["ciudad_normalizada"] = df_web["ciudad"].apply(normalizar_texto)

# Merge con Excel por id_cliente
df_consolidado = df_preliminar.merge(
    df_excel,
    on="id_cliente",
    how="left"
)

# Merge con tabla web por ciudad normalizada
df_consolidado = df_consolidado.merge(
    df_web[["ciudad_normalizada", "region", "macrozona"]],
    on="ciudad_normalizada",
    how="left"
)

print("\n===== DATAFRAME CONSOLIDADO =====")
display(df_consolidado.head())
print(f"Dimensiones del dataset consolidado: {df_consolidado.shape}")

# =========================================================
# 5) GUARDAR EL DATAFRAME CONSOLIDADO
# =========================================================
df_consolidado.to_csv(RUTA_CONSOLIDADO, index=False)

print(f"\nArchivo consolidado guardado en: {RUTA_CONSOLIDADO}")

===== DATAFRAME PRELIMINAR CARGADO DESDE CSV =====


,id_transaccion,id_cliente,fecha_transaccion,categoria_producto,cantidad,precio_unitario,descuento_pct,canal_venta,monto_compra,nombre,edad,ciudad,fecha_registro,categoria_favorita,ingreso_estimado
0,50001,1294,2024-01-20,Moda,4,70480.46,0.15,Web,239633.56,Diego Martínez,NaN,Santiago,2024-11-13,Libros,855625.0
1,50002,1024,2024-07-07,Hogar,5,9401.74,0.15,Marketplace,39957.39,María Díaz,50.0,Rancagua,2022-01-25,Deportes,935175.0
2,50003,1222,2024-05-31,Hogar,2,241376.52,0.05,Web,458615.39,Vicente Rojas,44.0,Temuco,2022-12-06,Hogar,1224321.0
3,50004,1100,2024-02-13,Libros,1,80071.35,0.00,App,80071.35,Benjamín González,33.0,Valparaíso,2024-09-24,Moda,676564.0
4,50005,1112,2024-06-21,Moda,5,66575.01,0.10,App,299587.54,Vicente Rojas,52.0,Santiago,2024-10-20,Libros,514152.0


Dimensiones del dataset preliminar: (1212, 15)

===== FUENTE EXCEL COMPLEMENTARIA =====


,id_cliente,segmento_cliente,canal_adquisicion,metodo_pago_preferido,score_fidelizacion
0,1294,Bronce,Orgánico,Tarjeta crédito,67.0
1,1024,Bronce,Orgánico,Tarjeta débito,98.0
2,1222,Oro,Referido,PayPal,83.0
3,1100,Premium,Referido,PayPal,96.0
4,1112,Premium,Orgánico,PayPal,91.0


Dimensiones del Excel complementario: (295, 5)

===== TABLA WEB LEÍDA CON read_html() =====


,ciudad,region,macrozona
0,Santiago,RegiÃ³n Metropolitana,Centro
1,ValparaÃ­so,RegiÃ³n de ValparaÃ­so,Centro
2,ConcepciÃ³n,RegiÃ³n del BiobÃ­o,Sur
3,La Serena,RegiÃ³n de Coquimbo,Norte
4,Antofagasta,RegiÃ³n de Antofagasta,Norte
5,Temuco,RegiÃ³n de La AraucanÃ­a,Sur
6,Rancagua,RegiÃ³n de O'Higgins,Centro
7,Puerto Montt,RegiÃ³n de Los Lagos,Sur


Dimensiones de la tabla web: (8, 3)

===== DATAFRAME CONSOLIDADO =====


,id_transaccion,id_cliente,fecha_transaccion,categoria_producto,cantidad,precio_unitario,descuento_pct,canal_venta,monto_compra,nombre,...,fecha_registro,categoria_favorita,ingreso_estimado,ciudad_normalizada,segmento_cliente,canal_adquisicion,metodo_pago_preferido,score_fidelizacion,region,macrozona
0,50001,1294,2024-01-20,Moda,4,70480.46,0.15,Web,239633.56,Diego Martínez,...,2024-11-13,Libros,855625.0,santiago,Bronce,Orgánico,Tarjeta crédito,67.0,RegiÃ³n Metropolitana,Centro
1,50002,1024,2024-07-07,Hogar,5,9401.74,0.15,Marketplace,39957.39,María Díaz,...,2022-01-25,Deportes,935175.0,rancagua,Bronce,Orgánico,Tarjeta débito,98.0,RegiÃ³n de O'Higgins,Centro
2,50003,1222,2024-05-31,Hogar,2,241376.52,0.05,Web,458615.39,Vicente Rojas,...,2022-12-06,Hogar,1224321.0,temuco,Oro,Referido,PayPal,83.0,RegiÃ³n de La AraucanÃ­a,Sur
3,50004,1100,2024-02-13,Libros,1,80071.35,0.00,App,80071.35,Benjamín González,...,2024-09-24,Moda,676564.0,valparaiso,Premium,Referido,PayPal,96.0,NaN,NaN
4,50005,1112,2024-06-21,Moda,5,66575.01,0.10,App,299587.54,Vicente Rojas,...,2024-10-20,Libros,514152.0,santiago,Premium,Orgánico,PayPal,91.0,RegiÃ³n Metropolitana,Centro


Dimensiones del dataset consolidado: (1212, 22)

Archivo consolidado guardado en: ../data/interim/dataset_consolidado.csv


## Lección 4 - Manejo de valores perdidos y outliers

In [6]:
import pandas as pd
import numpy as np

# =========================================================
# CONFIGURACIÓN DE RUTAS
# =========================================================
RUTA_CONSOLIDADO = "../data/interim/dataset_consolidado.csv"
RUTA_LIMPIO = "../data/interim/dataset_limpio.csv"

# =========================================================
# CARGA DEL DATASET CONSOLIDADO
# =========================================================
df_limpieza = pd.read_csv(RUTA_CONSOLIDADO)

# Reconversión de fechas
df_limpieza["fecha_transaccion"] = pd.to_datetime(df_limpieza["fecha_transaccion"])
df_limpieza["fecha_registro"] = pd.to_datetime(df_limpieza["fecha_registro"])

print("===== DIMENSIONES INICIALES =====")
print(df_limpieza.shape)

# =========================================================
# 1) REVISIÓN DE VALORES NULOS
# =========================================================
print("\n===== VALORES NULOS ANTES DEL TRATAMIENTO =====")
nulos_antes = df_limpieza.isna().sum()
display(nulos_antes[nulos_antes > 0].sort_values(ascending=False))

# =========================================================
# 2) TRATAMIENTO DE VALORES NULOS
# =========================================================
# Decisiones:
# - edad: imputación con mediana
# - descuento_pct: imputación con 0, asumiendo ausencia de descuento informado
# - score_fidelizacion: imputación con mediana
# - ciudad: categorización como "Desconocida"
# - region y macrozona: categorización como "Sin información"

mediana_edad = df_limpieza["edad"].median()
mediana_score = df_limpieza["score_fidelizacion"].median()

df_limpieza["edad"] = df_limpieza["edad"].fillna(mediana_edad)
df_limpieza["descuento_pct"] = df_limpieza["descuento_pct"].fillna(0)
df_limpieza["score_fidelizacion"] = df_limpieza["score_fidelizacion"].fillna(mediana_score)
df_limpieza["ciudad"] = df_limpieza["ciudad"].fillna("Desconocida")
df_limpieza["region"] = df_limpieza["region"].fillna("Sin información")
df_limpieza["macrozona"] = df_limpieza["macrozona"].fillna("Sin información")

print("\n===== VALORES NULOS DESPUÉS DEL TRATAMIENTO =====")
nulos_despues = df_limpieza.isna().sum()
display(nulos_despues[nulos_despues > 0].sort_values(ascending=False))

# =========================================================
# 3) DETECCIÓN DE OUTLIERS
# =========================================================
def calcular_limites_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    return limite_inferior, limite_superior

columnas_outliers = ["precio_unitario", "monto_compra"]
resumen_outliers = []

for columna in columnas_outliers:
    serie = df_limpieza[columna]

    # IQR
    li, ls = calcular_limites_iqr(serie)

    # Como son variables monetarias, no tiene sentido permitir valores negativos
    li = max(li, 0)

    # Z-score manual con numpy/pandas
    media = serie.mean()
    desviacion = serie.std(ddof=0)
    z_scores = (serie - media) / desviacion

    # Bandera de outliers
    df_limpieza[f"outlier_{columna}_iqr"] = (serie < li) | (serie > ls)
    df_limpieza[f"outlier_{columna}_zscore"] = np.abs(z_scores) > 3

    cantidad_iqr_antes = df_limpieza[f"outlier_{columna}_iqr"].sum()
    cantidad_z_antes = df_limpieza[f"outlier_{columna}_zscore"].sum()

    # Tratamiento: winsorización / capping
    df_limpieza[columna] = df_limpieza[columna].clip(lower=li, upper=ls)

    # Revisión posterior al tratamiento usando nuevamente IQR
    li_post, ls_post = calcular_limites_iqr(df_limpieza[columna])
    li_post = max(li_post, 0)
    outliers_iqr_despues = ((df_limpieza[columna] < li_post) | (df_limpieza[columna] > ls_post)).sum()

    resumen_outliers.append({
        "columna": columna,
        "outliers_iqr_antes": int(cantidad_iqr_antes),
        "outliers_zscore_antes": int(cantidad_z_antes),
        "outliers_iqr_despues": int(outliers_iqr_despues),
        "limite_superior_aplicado": round(ls, 2)
    })

df_resumen_outliers = pd.DataFrame(resumen_outliers)

print("\n===== RESUMEN DE OUTLIERS =====")
display(df_resumen_outliers)

# =========================================================
# 4) REVISIÓN GENERAL POST LIMPIEZA
# =========================================================
print("\n===== INFORMACIÓN GENERAL DEL DATASET LIMPIO =====")
df_limpieza.info()

print("\n===== ESTADÍSTICAS DESCRIPTIVAS DESPUÉS DE LA LIMPIEZA =====")
display(df_limpieza.describe(include="all"))

# =========================================================
# 5) GUARDAR DATASET LIMPIO
# =========================================================
df_limpieza.to_csv(RUTA_LIMPIO, index=False)

print(f"\nArchivo limpio guardado en: {RUTA_LIMPIO}")
print(f"Dimensiones del dataset limpio: {df_limpieza.shape}")

===== DIMENSIONES INICIALES =====
(1212, 22)

===== VALORES NULOS ANTES DEL TRATAMIENTO =====


macrozona             370
region                370
score_fidelizacion     39
edad                   38
ciudad                 29
ciudad_normalizada     29
descuento_pct          21
dtype: int64


===== VALORES NULOS DESPUÉS DEL TRATAMIENTO =====


ciudad_normalizada    29
dtype: int64


===== RESUMEN DE OUTLIERS =====


,columna,outliers_iqr_antes,outliers_zscore_antes,outliers_iqr_despues,limite_superior_aplicado
0,precio_unitario,6,6,0,359164.20
1,monto_compra,21,7,0,1107260.75



===== INFORMACIÓN GENERAL DEL DATASET LIMPIO =====
<class 'pandas.DataFrame'>
RangeIndex: 1212 entries, 0 to 1211
Data columns (total 26 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id_transaccion                  1212 non-null   int64         
 1   id_cliente                      1212 non-null   int64         
 2   fecha_transaccion               1212 non-null   datetime64[us]
 3   categoria_producto              1212 non-null   str           
 4   cantidad                        1212 non-null   int64         
 5   precio_unitario                 1212 non-null   float64       
 6   descuento_pct                   1212 non-null   float64       
 7   canal_venta                     1212 non-null   str           
 8   monto_compra                    1212 non-null   float64       
 9   nombre                          1212 non-null   str           
 10  edad                           

,id_transaccion,id_cliente,fecha_transaccion,categoria_producto,cantidad,precio_unitario,descuento_pct,canal_venta,monto_compra,nombre,...,segmento_cliente,canal_adquisicion,metodo_pago_preferido,score_fidelizacion,region,macrozona,outlier_precio_unitario_iqr,outlier_precio_unitario_zscore,outlier_monto_compra_iqr,outlier_monto_compra_zscore
count,1212.000000,1212.000000,1212,1212,1212.000000,1212.000000,1212.000000,1212,1.212000e+03,1212,...,1212,1212,1212,1212.000000,1212,1212,1212,1212,1212,1212
unique,NaN,NaN,NaN,9,NaN,NaN,NaN,6,NaN,174,...,7,5,4,NaN,7,4,2,2,2,2
top,NaN,NaN,NaN,Moda,NaN,NaN,NaN,App,NaN,Valentina Muñoz,...,Oro,Referido,Tarjeta crédito,NaN,Sin información,Sin información,False,False,False,False
freq,NaN,NaN,NaN,230,NaN,NaN,NaN,443,NaN,22,...,331,275,351,NaN,370,370,1206,1206,1191,1205
mean,50599.483498,1148.289604,2024-07-03 18:20:11.881188,NaN,3.014851,129962.330250,0.070957,NaN,3.614514e+05,NaN,...,NaN,NaN,NaN,71.775578,NaN,NaN,NaN,NaN,NaN,NaN
min,50001.000000,1001.000000,2024-01-01 00:00:00,NaN,1.000000,5116.950000,0.000000,NaN,4.895470e+03,NaN,...,NaN,NaN,NaN,40.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,50299.750000,1068.750000,2024-04-06 18:00:00,NaN,2.000000,71939.135000,0.000000,NaN,1.390858e+05,NaN,...,NaN,NaN,NaN,57.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,50599.500000,1148.500000,2024-07-11 00:00:00,NaN,3.000000,130214.170000,0.050000,NaN,2.919638e+05,NaN,...,NaN,NaN,NaN,71.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,50898.250000,1223.000000,2024-09-28 00:00:00,NaN,4.000000,186829.162500,0.100000,NaN,5.263558e+05,NaN,...,NaN,NaN,NaN,88.000000,NaN,NaN,NaN,NaN,NaN,NaN
max,51200.000000,1300.000000,2024-12-30 00:00:00,NaN,5.000000,359164.203750,0.200000,NaN,1.107261e+06,NaN,...,NaN,NaN,NaN,100.000000,NaN,NaN,NaN,NaN,NaN,NaN



Archivo limpio guardado en: ../data/interim/dataset_limpio.csv
Dimensiones del dataset limpio: (1212, 26)


## Lección 5 - Data Wrangling

In [8]:
import pandas as pd
import numpy as np
import unicodedata

# =========================================================
# CONFIGURACIÓN DE RUTAS
# =========================================================
RUTA_LIMPIO = "../data/interim/dataset_limpio.csv"
RUTA_WRANGLING = "../data/interim/dataset_wrangling.csv"

# =========================================================
# CARGA DEL DATASET LIMPIO
# =========================================================
df_wrangling = pd.read_csv(RUTA_LIMPIO)

# Reconversión de fechas
df_wrangling["fecha_transaccion"] = pd.to_datetime(df_wrangling["fecha_transaccion"])
df_wrangling["fecha_registro"] = pd.to_datetime(df_wrangling["fecha_registro"])

print("===== DIMENSIONES INICIALES =====")
print(df_wrangling.shape)

# =========================================================
# FUNCIONES AUXILIARES
# =========================================================
def normalizar_texto(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    return texto

def clasificar_fidelizacion(score):
    if score >= 85:
        return "Alta"
    elif score >= 65:
        return "Media"
    else:
        return "Baja"

# =========================================================
# 1) ELIMINAR DUPLICADOS
# =========================================================
filas_antes = df_wrangling.shape[0]

# Como duplicamos transacciones a propósito en la Lección 1,
# eliminamos duplicados por id_transaccion
df_wrangling = df_wrangling.drop_duplicates(subset=["id_transaccion"])

filas_despues = df_wrangling.shape[0]
duplicados_eliminados = filas_antes - filas_despues

print("\n===== DUPLICADOS =====")
print(f"Filas antes: {filas_antes}")
print(f"Filas después: {filas_despues}")
print(f"Duplicados eliminados: {duplicados_eliminados}")

# =========================================================
# 2) NORMALIZAR Y ESTANDARIZAR TEXTO
# =========================================================
# Ciudad
mapa_ciudades = {
    "santiago": "Santiago",
    "valparaiso": "Valparaíso",
    "concepcion": "Concepción",
    "la serena": "La Serena",
    "antofagasta": "Antofagasta",
    "temuco": "Temuco",
    "rancagua": "Rancagua",
    "puerto montt": "Puerto Montt",
    "desconocida": "Desconocida"
}

# Categoría producto / favorita
mapa_categorias = {
    "tecnologia": "Tecnología",
    "hogar": "Hogar",
    "moda": "Moda",
    "deportes": "Deportes",
    "belleza": "Belleza",
    "libros": "Libros"
}

# Canal
mapa_canales = {
    "web": "Web",
    "app": "App",
    "marketplace": "Marketplace"
}

# Segmento
mapa_segmentos = {
    "bronce": "Bronce",
    "plata": "Plata",
    "oro": "Oro",
    "premium": "Premium"
}

df_wrangling["ciudad"] = df_wrangling["ciudad"].apply(normalizar_texto).map(mapa_ciudades)
df_wrangling["categoria_favorita"] = df_wrangling["categoria_favorita"].apply(normalizar_texto).map(mapa_categorias)
df_wrangling["categoria_producto"] = df_wrangling["categoria_producto"].apply(normalizar_texto).map(mapa_categorias)
df_wrangling["canal_venta"] = df_wrangling["canal_venta"].apply(normalizar_texto).map(mapa_canales)
df_wrangling["segmento_cliente"] = df_wrangling["segmento_cliente"].apply(normalizar_texto).map(mapa_segmentos)

# Si algo no matchea, rellenamos con categoría genérica
df_wrangling["ciudad"] = df_wrangling["ciudad"].fillna("Desconocida")
df_wrangling["categoria_favorita"] = df_wrangling["categoria_favorita"].fillna("Sin categoría")
df_wrangling["categoria_producto"] = df_wrangling["categoria_producto"].fillna("Sin categoría")
df_wrangling["canal_venta"] = df_wrangling["canal_venta"].fillna("Sin canal")
df_wrangling["segmento_cliente"] = df_wrangling["segmento_cliente"].fillna("Sin segmento")

# =========================================================
# 3) TRANSFORMAR TIPOS DE DATOS
# =========================================================
df_wrangling["edad"] = df_wrangling["edad"].round().astype(int)
df_wrangling["score_fidelizacion"] = df_wrangling["score_fidelizacion"].round().astype(int)
df_wrangling["cantidad"] = df_wrangling["cantidad"].astype(int)

columnas_categoricas = [
    "ciudad",
    "categoria_favorita",
    "categoria_producto",
    "canal_venta",
    "segmento_cliente",
    "metodo_pago_preferido",
    "region",
    "macrozona"
]

for col in columnas_categoricas:
    df_wrangling[col] = df_wrangling[col].astype("category")

# =========================================================
# 4) CREAR NUEVAS COLUMNAS CALCULADAS
# =========================================================
df_wrangling["monto_bruto"] = df_wrangling["cantidad"] * df_wrangling["precio_unitario"]
df_wrangling["monto_descuento"] = df_wrangling["monto_bruto"] * df_wrangling["descuento_pct"]
df_wrangling["antiguedad_cliente_dias"] = (
    df_wrangling["fecha_transaccion"] - df_wrangling["fecha_registro"]
).dt.days

# =========================================================
# 5) APPLY, MAP Y LAMBDA
# =========================================================
# apply()
df_wrangling["nivel_fidelizacion"] = df_wrangling["score_fidelizacion"].apply(clasificar_fidelizacion)

# map()
mapa_macrozona_num = {
    "Norte": 1,
    "Centro": 2,
    "Sur": 3,
    "Sin información": 0
}
df_wrangling["codigo_macrozona"] = df_wrangling["macrozona"].astype(str).map(mapa_macrozona_num)

# lambda()
df_wrangling["compra_con_descuento"] = df_wrangling["descuento_pct"].apply(
    lambda x: "Sí" if x > 0 else "No"
)

# =========================================================
# 6) NORMALIZACIÓN Y DISCRETIZACIÓN
# =========================================================
# Normalización Min-Max del score de fidelización
score_min = df_wrangling["score_fidelizacion"].min()
score_max = df_wrangling["score_fidelizacion"].max()

df_wrangling["score_fidelizacion_norm"] = (
    (df_wrangling["score_fidelizacion"] - score_min) / (score_max - score_min)
)

# Discretización de edad
df_wrangling["grupo_edad"] = pd.cut(
    df_wrangling["edad"],
    bins=[17, 29, 44, 59, 100],
    labels=["18-29", "30-44", "45-59", "60+"]
)

# Discretización del monto_compra
df_wrangling["rango_monto_compra"] = pd.qcut(
    df_wrangling["monto_compra"],
    q=4,
    labels=["Bajo", "Medio", "Alto", "Muy alto"]
)

# =========================================================
# 7) REVISIÓN FINAL
# =========================================================
print("\n===== VISTA PRELIMINAR DEL DATASET WRANGLING =====")
display(df_wrangling.head())

print("\n===== INFORMACIÓN GENERAL =====")
df_wrangling.info()

print("\n===== RESUMEN DE CATEGORÍAS LIMPIAS =====")
print("Categorías de producto:", df_wrangling["categoria_producto"].astype(str).unique())
print("Canales de venta:", df_wrangling["canal_venta"].astype(str).unique())
print("Segmentos:", df_wrangling["segmento_cliente"].astype(str).unique())

# =========================================================
# 8) GUARDAR DATASET OPTIMIZADO
# =========================================================
df_wrangling.to_csv(RUTA_WRANGLING, index=False)

print(f"\nArchivo wrangling guardado en: {RUTA_WRANGLING}")
print(f"Dimensiones del dataset wrangling: {df_wrangling.shape}")

===== DIMENSIONES INICIALES =====
(1212, 26)

===== DUPLICADOS =====
Filas antes: 1212
Filas después: 1200
Duplicados eliminados: 12

===== VISTA PRELIMINAR DEL DATASET WRANGLING =====


,id_transaccion,id_cliente,fecha_transaccion,categoria_producto,cantidad,precio_unitario,descuento_pct,canal_venta,monto_compra,nombre,...,outlier_monto_compra_zscore,monto_bruto,monto_descuento,antiguedad_cliente_dias,nivel_fidelizacion,codigo_macrozona,compra_con_descuento,score_fidelizacion_norm,grupo_edad,rango_monto_compra
0,50001,1294,2024-01-20,Moda,4,70480.46,0.15,Web,239633.56,Diego Martínez,...,False,281921.84,42288.276,-298,Media,2,Sí,0.450000,30-44,Medio
1,50002,1024,2024-07-07,Hogar,5,9401.74,0.15,Marketplace,39957.39,María Díaz,...,False,47008.70,7051.305,894,Alta,2,Sí,0.966667,45-59,Bajo
2,50003,1222,2024-05-31,Hogar,2,241376.52,0.05,Web,458615.39,Vicente Rojas,...,False,482753.04,24137.652,542,Media,3,Sí,0.716667,30-44,Alto
3,50004,1100,2024-02-13,Libros,1,80071.35,0.00,App,80071.35,Benjamín González,...,False,80071.35,0.000,-224,Alta,0,No,0.933333,30-44,Bajo
4,50005,1112,2024-06-21,Moda,5,66575.01,0.10,App,299587.54,Vicente Rojas,...,False,332875.05,33287.505,-121,Alta,2,Sí,0.850000,45-59,Alto



===== INFORMACIÓN GENERAL =====
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 35 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id_transaccion                  1200 non-null   int64         
 1   id_cliente                      1200 non-null   int64         
 2   fecha_transaccion               1200 non-null   datetime64[us]
 3   categoria_producto              1200 non-null   category      
 4   cantidad                        1200 non-null   int64         
 5   precio_unitario                 1200 non-null   float64       
 6   descuento_pct                   1200 non-null   float64       
 7   canal_venta                     1200 non-null   category      
 8   monto_compra                    1200 non-null   float64       
 9   nombre                          1200 non-null   str           
 10  edad                            1200 non-null   in

## Lección 6 - Agrupamiento y pivoteo de datos

In [10]:
import pandas as pd
import numpy as np
import unicodedata

# =========================================================
# CONFIGURACIÓN DE RUTAS
# =========================================================
RUTA_WRANGLING = "../data/interim/dataset_wrangling.csv"
RUTA_FINAL_CSV = "../data/processed/dataset_final.csv"
RUTA_FINAL_XLSX = "../data/processed/dataset_final.xlsx"
RUTA_RESUMEN_CSV = "../outputs/resumen_comercial.csv"

# =========================================================
# FUNCIÓN AUXILIAR
# =========================================================
def normalizar_texto(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    return texto

# =========================================================
# 1) CARGAR DATASET WRANGLING
# =========================================================
df_final = pd.read_csv(RUTA_WRANGLING)

df_final["fecha_transaccion"] = pd.to_datetime(df_final["fecha_transaccion"])
df_final["fecha_registro"] = pd.to_datetime(df_final["fecha_registro"])

print("===== DIMENSIONES INICIALES =====")
print(df_final.shape)

# =========================================================
# 2) AJUSTE FINAL DE COLUMNA AUXILIAR
# =========================================================
# Recalculamos ciudad_normalizada a partir de la ciudad ya limpia
df_final["ciudad_normalizada"] = df_final["ciudad"].astype(str).apply(normalizar_texto)

# =========================================================
# 3) GROUPBY: MÉTRICAS RESUMIDAS POR CLIENTE
# =========================================================
resumen_clientes = (
    df_final
    .groupby("id_cliente", as_index=False)
    .agg(
        total_transacciones=("id_transaccion", "count"),
        gasto_total_cliente=("monto_compra", "sum"),
        ticket_promedio_cliente=("monto_compra", "mean"),
        cantidad_total_productos=("cantidad", "sum"),
        descuento_promedio_cliente=("descuento_pct", "mean"),
        score_fidelizacion_promedio=("score_fidelizacion", "mean")
    )
)

print("\n===== RESUMEN POR CLIENTE =====")
display(resumen_clientes.head())

# =========================================================
# 4) MERGE: INCORPORAR MÉTRICAS RESUMIDAS AL DATASET FINAL
# =========================================================
df_final = df_final.merge(
    resumen_clientes,
    on="id_cliente",
    how="left"
)

print("\n===== DATASET FINAL CON MÉTRICAS AGREGADAS =====")
display(df_final.head())

# =========================================================
# 5) GROUPBY: RESUMEN COMERCIAL POR CANAL
# =========================================================
resumen_canal = (
    df_final
    .groupby("canal_venta", as_index=False)
    .agg(
        transacciones=("id_transaccion", "count"),
        ventas_totales=("monto_compra", "sum"),
        ticket_promedio=("monto_compra", "mean")
    )
)

resumen_canal["tipo_resumen"] = "Canal"
resumen_canal = resumen_canal.rename(columns={"canal_venta": "dimension"})

# =========================================================
# 6) GROUPBY: RESUMEN COMERCIAL POR CATEGORÍA
# =========================================================
resumen_categoria = (
    df_final
    .groupby("categoria_producto", as_index=False)
    .agg(
        transacciones=("id_transaccion", "count"),
        ventas_totales=("monto_compra", "sum"),
        ticket_promedio=("monto_compra", "mean")
    )
)

resumen_categoria["tipo_resumen"] = "Categoria"
resumen_categoria = resumen_categoria.rename(columns={"categoria_producto": "dimension"})

# =========================================================
# 7) CONCAT: UNIR RESÚMENES
# =========================================================
resumen_comercial = pd.concat(
    [resumen_canal, resumen_categoria],
    ignore_index=True
)

print("\n===== RESUMEN COMERCIAL CONCATENADO =====")
display(resumen_comercial)

# =========================================================
# 8) PIVOT: VENTAS POR CATEGORÍA Y CANAL
# =========================================================
tabla_pivote = df_final.pivot_table(
    index="categoria_producto",
    columns="canal_venta",
    values="monto_compra",
    aggfunc="sum",
    fill_value=0
)

print("\n===== TABLA PIVOTE =====")
display(tabla_pivote)

# =========================================================
# 9) MELT: PASAR LA TABLA PIVOTE A FORMATO LARGO
# =========================================================
tabla_pivote_reset = tabla_pivote.reset_index()

tabla_melt = tabla_pivote_reset.melt(
    id_vars="categoria_producto",
    var_name="canal_venta",
    value_name="ventas_totales"
)

print("\n===== TABLA EN FORMATO LARGO (MELT) =====")
display(tabla_melt.head(10))

# =========================================================
# 10) ORDEN FINAL DEL DATASET
# =========================================================
columnas_principales = [
    "id_transaccion", "id_cliente", "fecha_transaccion", "fecha_registro",
    "nombre", "edad", "grupo_edad", "ciudad", "region", "macrozona",
    "segmento_cliente", "canal_adquisicion", "canal_venta",
    "categoria_producto", "categoria_favorita", "cantidad",
    "precio_unitario", "descuento_pct", "monto_bruto",
    "monto_descuento", "monto_compra", "ingreso_estimado",
    "score_fidelizacion", "nivel_fidelizacion",
    "total_transacciones", "gasto_total_cliente", "ticket_promedio_cliente",
    "cantidad_total_productos", "descuento_promedio_cliente",
    "score_fidelizacion_promedio"
]

otras_columnas = [col for col in df_final.columns if col not in columnas_principales]
df_final = df_final[columnas_principales + otras_columnas]

# =========================================================
# 11) EXPORTACIÓN FINAL
# =========================================================
df_final.to_csv(RUTA_FINAL_CSV, index=False)
resumen_comercial.to_csv(RUTA_RESUMEN_CSV, index=False)

with pd.ExcelWriter(RUTA_FINAL_XLSX, engine="openpyxl") as writer:
    df_final.to_excel(writer, sheet_name="dataset_final", index=False)
    resumen_clientes.to_excel(writer, sheet_name="resumen_clientes", index=False)
    resumen_comercial.to_excel(writer, sheet_name="resumen_comercial", index=False)
    tabla_pivote.to_excel(writer, sheet_name="pivot_ventas")
    tabla_melt.to_excel(writer, sheet_name="melt_ventas", index=False)

print(f"\nArchivo final CSV guardado en: {RUTA_FINAL_CSV}")
print(f"Archivo final Excel guardado en: {RUTA_FINAL_XLSX}")
print(f"Archivo resumen CSV guardado en: {RUTA_RESUMEN_CSV}")
print(f"Dimensiones del dataset final: {df_final.shape}")

===== DIMENSIONES INICIALES =====
(1200, 35)

===== RESUMEN POR CLIENTE =====


,id_cliente,total_transacciones,gasto_total_cliente,ticket_promedio_cliente,cantidad_total_productos,descuento_promedio_cliente,score_fidelizacion_promedio
0,1001,5,2.619020e+06,523804.082000,18,0.060000,49.0
1,1002,7,3.092160e+06,441737.182857,21,0.042857,92.0
2,1003,6,2.850330e+06,475055.070000,18,0.058333,47.0
3,1004,1,1.758452e+05,175845.220000,2,0.200000,64.0
4,1005,6,2.015988e+06,335998.042708,14,0.075000,64.0



===== DATASET FINAL CON MÉTRICAS AGREGADAS =====


,id_transaccion,id_cliente,fecha_transaccion,categoria_producto,cantidad,precio_unitario,descuento_pct,canal_venta,monto_compra,nombre,...,compra_con_descuento,score_fidelizacion_norm,grupo_edad,rango_monto_compra,total_transacciones,gasto_total_cliente,ticket_promedio_cliente,cantidad_total_productos,descuento_promedio_cliente,score_fidelizacion_promedio
0,50001,1294,2024-01-20,Moda,4,70480.46,0.15,Web,239633.56,Diego Martínez,...,Sí,0.450000,30-44,Medio,6,2602462.58,433743.763333,23,0.083333,67.0
1,50002,1024,2024-07-07,Hogar,5,9401.74,0.15,Marketplace,39957.39,María Díaz,...,Sí,0.966667,45-59,Bajo,9,2557087.22,284120.802222,25,0.088889,98.0
2,50003,1222,2024-05-31,Hogar,2,241376.52,0.05,Web,458615.39,Vicente Rojas,...,Sí,0.716667,30-44,Alto,6,2204779.97,367463.328333,17,0.066667,83.0
3,50004,1100,2024-02-13,Libros,1,80071.35,0.00,App,80071.35,Benjamín González,...,No,0.933333,30-44,Bajo,6,1147085.78,191180.963333,12,0.066667,96.0
4,50005,1112,2024-06-21,Moda,5,66575.01,0.10,App,299587.54,Vicente Rojas,...,Sí,0.850000,45-59,Alto,2,1138852.23,569426.115000,9,0.075000,91.0



===== RESUMEN COMERCIAL CONCATENADO =====


...


===== TABLA PIVOTE =====


canal_venta,App,Marketplace,Web
categoria_producto,,,
Belleza,2.639235e+07,1.760841e+07,2.123899e+07
Deportes,2.373001e+07,1.980057e+07,2.183138e+07
Hogar,2.390195e+07,2.138071e+07,1.979344e+07
Libros,2.417276e+07,1.731712e+07,2.898513e+07
Moda,3.509674e+07,3.663042e+07,2.179768e+07
Tecnología,2.960261e+07,2.307658e+07,2.059998e+07



===== TABLA EN FORMATO LARGO (MELT) =====


,categoria_producto,canal_venta,ventas_totales
0,Belleza,App,2.639235e+07
1,Deportes,App,2.373001e+07
2,Hogar,App,2.390195e+07
3,Libros,App,2.417276e+07
4,Moda,App,3.509674e+07
5,Tecnología,App,2.960261e+07
6,Belleza,Marketplace,1.760841e+07
7,Deportes,Marketplace,1.980057e+07
8,Hogar,Marketplace,2.138071e+07
9,Libros,Marketplace,1.731712e+07



Archivo final CSV guardado en: ../data/processed/dataset_final.csv
Archivo final Excel guardado en: ../data/processed/dataset_final.xlsx
Archivo resumen CSV guardado en: ../outputs/resumen_comercial.csv
Dimensiones del dataset final: (1200, 41)
